## AURN data cleaning pipeline

Loads DEFRA AURN hourly CSV exports, applies quality-flag filtering,
forward-fills short gaps (<=3 h per city), removes rolling z-score
outliers, and saves the cleaned dataset to `data/processed/`.

In [1]:
import pathlib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)
pd.set_option('display.max_columns', 20)

In [2]:
# Cell 1 -- load all CSVs from data/raw/ and combine
RAW_DIR = pathlib.Path('../data/raw')


def _site_map(path: pathlib.Path) -> dict:
    """Return {col_index: station_name} from the DEFRA preamble 'Site Name' row."""
    import csv as _csv
    with open(path, encoding='utf-8-sig', errors='replace') as f:
        for line in f:
            if line.startswith('Site Name'):
                parts = next(_csv.reader([line]))
                return {
                    i: v.strip()
                    for i, v in enumerate(parts)
                    if v.strip() not in ('Site Name', '')
                }
    raise ValueError(f'No Site Name row found in {path}')


def _header_row_idx(path: pathlib.Path) -> int:
    """Return 0-indexed line number of the Date/Time column-header line."""
    with open(path, encoding='utf-8-sig', errors='replace') as f:
        for i, line in enumerate(f):
            if line.startswith('Date,'):
                return i
    raise ValueError(f'No Date header row found in {path}')


def load_defra_csv(path: pathlib.Path) -> pd.DataFrame:
    """
    Parse a wide-format DEFRA AURN export into a tidy long DataFrame.

    Each station occupies 6 contiguous columns after Date/Time:
        [O3_val, O3_flag, NO2_val, NO2_flag, PM2.5_val, PM2.5_flag]
    """
    site_map   = _site_map(path)
    header_row = _header_row_idx(path)

    data = pd.read_csv(
        path, skiprows=header_row, header=0,
        dtype=str, low_memory=False,
    )

    frames = []
    for col_start, site_name in sorted(site_map.items()):
        block = data.iloc[
            :, [0, 1,
                col_start,     col_start + 1,
                col_start + 2, col_start + 3,
                col_start + 4, col_start + 5]
        ].copy()
        block.columns = ['date', 'time',
                         'o3',   'o3_flag',
                         'no2',  'no2_flag',
                         'pm25', 'pm25_flag']
        block.insert(0, 'city', site_name)
        frames.append(block)

    return pd.concat(frames, ignore_index=True)


csv_files = sorted(RAW_DIR.glob('*.csv'))
df_raw    = pd.concat([load_defra_csv(f) for f in csv_files], ignore_index=True)

print(f'Loaded {len(df_raw):,} rows from {len(csv_files)} file(s)')
print('Cities:', sorted(df_raw['city'].unique()))
df_raw.head(3)

Loaded 87,725 rows from 1 file(s)
Cities: ['Birmingham A4540 Roadside', 'Edinburgh St Leonards', 'Leeds Centre', 'London Marylebone Road', 'Manchester Piccadilly']


,city,date,time,o3,o3_flag,no2,no2_flag,pm25,pm25_flag
0,Birmingham A4540 Roadside,2023-01-01,01:00:00,61.33451,V ugm-3,14.17608,V ugm-3,7.123,V ugm-3 (Ref.eq)
1,Birmingham A4540 Roadside,2023-01-01,02:00:00,65.79158,V ugm-3,10.04362,V ugm-3,4.057,V ugm-3 (Ref.eq)
2,Birmingham A4540 Roadside,2023-01-01,03:00:00,66.40692,V ugm-3,11.48356,V ugm-3,4.363,V ugm-3 (Ref.eq)


In [3]:
# Cell 2 -- parse datetime and set as index
df = df_raw.copy()

# Pollutant columns arrive as str; cast to float
for col in ['o3', 'no2', 'pm25']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# DEFRA uses '24:00:00' for the midnight hour-ending (= next day 00:00).
# Replace before parsing, then shift those rows forward by one day.
time_col      = df['time'].str.strip()
midnight_mask = time_col == '24:00:00'
time_col      = time_col.str.replace('24:00:00', '00:00:00', regex=False)

df['datetime'] = pd.to_datetime(
    df['date'].str.strip() + ' ' + time_col,
    format='%Y-%m-%d %H:%M:%S',
)
df.loc[midnight_mask, 'datetime'] += pd.Timedelta(days=1)

df = (
    df.drop(columns=['date', 'time'])
      .set_index('datetime')
      .sort_values(['city', 'datetime'])
)

print(f'Date range : {df.index.min()}  to  {df.index.max()}')
print(f'Shape      : {df.shape}')
df.head(3)

Date range : 2023-01-01 01:00:00  to  2025-01-01 00:00:00
Shape      : (87725, 7)


,city,o3,o3_flag,no2,no2_flag,pm25,pm25_flag
datetime,,,,,,,
2023-01-01 01:00:00,Birmingham A4540 Roadside,61.33451,V ugm-3,14.17608,V ugm-3,7.123,V ugm-3 (Ref.eq)
2023-01-01 02:00:00,Birmingham A4540 Roadside,65.79158,V ugm-3,10.04362,V ugm-3,4.057,V ugm-3 (Ref.eq)
2023-01-01 03:00:00,Birmingham A4540 Roadside,66.40692,V ugm-3,11.48356,V ugm-3,4.363,V ugm-3 (Ref.eq)


In [4]:
# Cell 3 -- handle DEFRA quality flags
# Status strings look like: 'V ugm-3', 'P ugm-3 (Ref.eq)', 'N ugm-3', 'S ugm-3 (BAM)'
# Leading letter:
#   V = Verified               -> keep
#   P = Provisionally Verified -> keep
#   N = Not Verified           -> NaN
#   S = Suspect                -> NaN

VALID_FLAGS = {'V', 'P'}

for val_col, flag_col in [('o3',  'o3_flag'),
                           ('no2', 'no2_flag'),
                           ('pm25','pm25_flag')]:
    flag_letter = df[flag_col].str.strip().str[0].str.upper()
    df.loc[~flag_letter.isin(VALID_FLAGS), val_col] = np.nan

df = df.drop(columns=['o3_flag', 'no2_flag', 'pm25_flag'])

# Snapshot missing fractions BEFORE gap-fill / outlier removal (used in Cell 6)
missing_before = df[['o3', 'no2', 'pm25']].isna().mean()

print('Missing % after quality-flag filtering (before gap-fill and outlier removal):')
print((missing_before * 100).round(2).to_string())
df.head(3)

Missing % after quality-flag filtering (before gap-fill and outlier removal):
o3      4.72
no2     4.93
pm25    7.14


,city,o3,no2,pm25
datetime,,,,
2023-01-01 01:00:00,Birmingham A4540 Roadside,61.33451,14.17608,7.123
2023-01-01 02:00:00,Birmingham A4540 Roadside,65.79158,10.04362,4.057
2023-01-01 03:00:00,Birmingham A4540 Roadside,66.40692,11.48356,4.363


In [5]:
# Cell 4 -- forward-fill gaps <= 3 consecutive hours per city
# Runs longer than 3 h remain NaN.

df[['o3', 'no2', 'pm25']] = (
    df.groupby('city')[['o3', 'no2', 'pm25']]
    .transform(lambda s: s.ffill(limit=3))
)

filled = (
    df[['o3', 'no2', 'pm25']].notna().sum()
    - df_raw[['o3', 'no2', 'pm25']].apply(pd.to_numeric, errors='coerce').notna().sum()
)
print('Rows recovered by forward-fill (limit=3 h):')
print(filled.to_string())

Rows recovered by forward-fill (limit=3 h):
o3      508
no2     499
pm25    286


In [6]:
# Cell 5 -- rolling z-score outlier removal (window=720 h, threshold=3.5)
WINDOW    = 720   # hours (~30 days)
THRESHOLD = 3.5


def _remove_outliers(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    for col in ['o3', 'no2', 'pm25']:
        roll   = group[col].rolling(window=WINDOW, center=True, min_periods=WINDOW // 2)
        mu     = roll.mean()
        sigma  = roll.std().replace(0, np.nan)
        z      = (group[col] - mu) / sigma
        group.loc[z.abs() > THRESHOLD, col] = np.nan
    return group


import warnings as _w
with _w.catch_warnings():
    _w.simplefilter('ignore', FutureWarning)
    df = df.groupby('city', group_keys=False).apply(_remove_outliers)
print(f'Outlier removal done  (window={WINDOW} h, threshold |z|>{THRESHOLD} -> NaN)')

Outlier removal done  (window=720 h, threshold |z|>3.5 -> NaN)


In [7]:
# Cell 6 -- missing % before vs after cleaning
missing_after = df[['o3', 'no2', 'pm25']].isna().mean()

summary = pd.DataFrame({
    'before (%)':     (missing_before  * 100).round(2),
    'after  (%)':     (missing_after   * 100).round(2),
    'recovered (pp)': ((missing_before - missing_after) * 100).round(2),
})
print('=== Missing data: before vs after cleaning ===')
print(summary.to_string())

=== Missing data: before vs after cleaning ===
      before (%)  after  (%)  recovered (pp)
o3          4.72        4.25            0.47
no2         4.93        4.81            0.12
pm25        7.14        7.49           -0.35


In [8]:
# Cell 7 -- save to data/processed/aurn_cleaned.csv
OUT_PATH = pathlib.Path('../data/processed/aurn_cleaned.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_PATH)
print(f'Saved {len(df):,} rows x {len(df.columns)} columns to {OUT_PATH}')
df.tail(3)

Saved 87,725 rows x 4 columns to ..\data\processed\aurn_cleaned.csv


,city,o3,no2,pm25
datetime,,,,
2024-12-31 23:00:00,Manchester Piccadilly,62.81466,18.76406,3.679
2025-01-01 00:00:00,Manchester Piccadilly,61.11831,18.23535,3.373
NaT,Manchester Piccadilly,61.11831,18.23535,3.373
